# Axolotl telencephalon — CellSTIC tutorial
**Dataset:** Spatial telencephalon; per-batch under `data/axolotl_telencephalon/<batch>/`; split from combined `raw/development.h5ad` / `regeneration.h5ad`. Ligand–receptor map `data/axolotl_telencephalon/LR.csv`.


## 1. Repository path


In [ ]:
from pathlib import Path
import sys
import torch

# Add project root to path
try:
    project_root = Path(__file__).resolve().parent.parent
except NameError:
    _cwd = Path.cwd()
    project_root = _cwd if (_cwd / "model").is_dir() else _cwd.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root.resolve()}")


## 2. Imports


In [ ]:
from utils.tools.seed_utils import set_global_seed
import anndata as ad
from typing import Dict

from model.cellstic import CellSTIC
from pipeline.analyzer import TimeSequenceAnalysis
from pipeline.evaluator import CellSTICEvaluator
from pipeline.trainer import CellSTICTrainer
from utils.loader import load_axolotl_telencephalon
from utils.metrics import MetricsComputer
from utils.train import ModelUtils, load_config

set_global_seed()  # once before train/eval; pipeline does not set RNG here


## 3. Configuration

Edit switches and paths before running downstream sections.


In [ ]:
# --- Run switches ---
RUN_SPLIT_DATA = False
RUN_SINGLE_BATCH_PIPELINE = False
RUN_SINGLE_BATCH_VIS = False
RUN_TIME_SERIES_ANALYSIS = True

# Train when running the single-batch pipeline
RUN_TRAIN = False

# Example batch for the single-batch workflow
BATCH_NAME = "develop_44"

# Optional visualization label key
COLOR_KEY = "Annotation"

# Choose one temporal program for stage-wise analysis
TIME_MODE = "development"   # choices: "development", "regeneration"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Base dataset folder
dataset_root = project_root / "data" / "axolotl_telencephalon"

# Current batch work directory
work_dir = dataset_root / BATCH_NAME
output_path = work_dir / "output"
raw_path = work_dir / "raw"
model_path = work_dir / "model"
preprocess_path = work_dir / "preprocess"
config_path = work_dir / "config" / "axolotl_telencephalon_config.yaml"
lr_path = work_dir.parent / "LR.csv"

for _p in [output_path, raw_path, model_path, preprocess_path]:
    _p.mkdir(parents=True, exist_ok=True)

print("dataset_root =", dataset_root)
print("work_dir =", work_dir)
print("RUN_SINGLE_BATCH_PIPELINE =", RUN_SINGLE_BATCH_PIPELINE)
print("Device:", device)
print("raw_path =", raw_path)
print("preprocess_path =", preprocess_path)
print("model_path =", model_path)
print("output_path =", output_path)
print("config_path =", config_path)
print("lr_path =", lr_path)


## 4. Optional data splitting

Use this only when you start from the combined raw files under:

- `data/axolotl_telencephalon/raw/development.h5ad`
- `data/axolotl_telencephalon/raw/regeneration.h5ad`

This step splits each combined file into per-batch folders such as `develop_44` or `regeneration_30`.


In [ ]:
_BATCH_TO_DIR_DEVELOPMENT: Dict[str, str] = {
    "Batch1_Adult_telencephalon_rep2_DP8400015234BLA3_1": "develop_Adult",
    "Injury_control_FP200000239BL_E3": "develop_Juvenile",
    "Meta_telencephalon_rep1_DP8400015234BLB2_1": "develop_Metamorphosed",
    "Stage44_telencephalon_rep2_FP200000239BL_E4": "develop_44",
    "Stage54_telencephalon_rep2_DP8400015649BRD6_2": "develop_54",
    "Stage57_telencephalon_rep2_DP8400015649BRD5_1": "develop_57",
}

_BATCH_TO_DIR_REGENERATION: Dict[str, str] = {
    "Injury_2DPI_rep1_SS200000147BL_D5": "regeneration_2",
    "Injury_5DPI_rep1_SS200000147BL_D2": "regeneration_5",
    "Injury_10DPI_rep1_SS200000147BL_B5": "regeneration_10",
    "Injury_15DPI_rep4_FP200000266TR_E4": "regeneration_15",
    "Injury_20DPI_rep2_SS200000147BL_B4": "regeneration_20",
    "Injury_30DPI_rep2_FP200000264BL_A6": "regeneration_30",
    "Injury_60DPI_rep3_FP200000264BL_A6": "regeneration_60",
    "Injury_control_FP200000239BL_E3": "regeneration_control",
}

_SPLIT_MAP_BY_H5AD_NAME: Dict[str, Dict[str, str]] = {
    "development.h5ad": _BATCH_TO_DIR_DEVELOPMENT,
    "regeneration.h5ad": _BATCH_TO_DIR_REGENERATION,
}

if RUN_SPLIT_DATA:
    src_dir = dataset_root / "raw"
    dst_base = dataset_root

    h5ad_files = sorted(src_dir.glob("*.h5ad"))
    if not h5ad_files:
        raise FileNotFoundError(f"No .h5ad files found under {src_dir}")

    for h5ad_path in h5ad_files:
        key = h5ad_path.name.lower()
        batch_map = _SPLIT_MAP_BY_H5AD_NAME.get(key)
        if batch_map is None:
            raise ValueError(
                f"{h5ad_path.name}: no split mapping defined. "
                "Add it to _SPLIT_MAP_BY_H5AD_NAME."
            )

        adata = ad.read_h5ad(h5ad_path, backed="r")
        if "Batch" not in adata.obs.columns:
            raise KeyError(f"{h5ad_path.name}: obs does not contain 'Batch' column")

        batch_series = adata.obs["Batch"].astype(str)

        def _to_dir(batch_name: str) -> str:
            if batch_name not in batch_map:
                raise KeyError(
                    f"{h5ad_path.name}: unknown Batch {batch_name!r}. "
                    f"Known keys: {sorted(batch_map.keys())}"
                )
            return batch_map[batch_name]

        batch_folders = batch_series.map(_to_dir)

        for folder in sorted(batch_folders.unique()):
            mask = (batch_folders == folder).to_numpy()
            if mask.sum() == 0:
                continue

            out_dir = dst_base / folder / "raw"
            out_dir.mkdir(parents=True, exist_ok=True)
            out_path = out_dir / h5ad_path.name

            sub = adata[mask].to_memory()
            sub.write_h5ad(out_path)
            print(f"Saved: {out_path}")


## 5. Load configuration and data

Load YAML and axolotl telencephalon `AnnData` plus ligand–receptor map for `BATCH_NAME`.


In [ ]:
if RUN_SINGLE_BATCH_PIPELINE:
    # YAML experiment + model hyperparameters
    config = load_config(config_path)

    # Cached preprocess + LR graph metadata for this batch
    rna_adata, ligand_receptor_map = load_axolotl_telencephalon(
        raw_path,
        preprocess_path,
        use_cache=True,
        lr_path=lr_path,
    )
else:
    print("Skip load (set RUN_SINGLE_BATCH_PIPELINE=True to run §5–§7).")

## 6. Train or load model

Train `CellSTIC` when `RUN_TRAIN` is True; otherwise load weights from `model_path`.


In [ ]:
if RUN_SINGLE_BATCH_PIPELINE:
    if RUN_TRAIN:
        model = CellSTIC(config.model, device)
        CellSTICTrainer(
            model,
            config,
            model_path=model_path,
            device=device,
            ligand_receptor_map=ligand_receptor_map,
        ).train(
            modality_datas=[rna_adata],
            is_train_ccc=True,
            is_train_feature=True,
        )
    else:
        model = ModelUtils.load_model(
            model_path=model_path,
            config=config,
            device=device,
        )
else:
    print("Skip train/load.")

## 7. Evaluate features and CCC

Fused-feature metrics, then ligand–receptor edge probabilities on the spatial graph.


In [ ]:
if RUN_SINGLE_BATCH_PIPELINE:
    evaluator = CellSTICEvaluator(
        model,
        config,
        ligand_receptor_map=ligand_receptor_map,
        model_path=model_path,
        output_path=output_path,
        device=device,
    )

    # Clustering / silhouette on integrated embeddings
    evaluator.evaluate_mutiple_feature(
        modality_datas=[rna_adata],
        auto_n_clusters=7,
    )

    pos_edge_probs_np, edge_type_map, m0 = evaluator.evaluate_ccc_precision(
        modality_datas=[rna_adata],
    )

    print("Single-batch evaluation finished.")
else:
    print("Skip evaluation.")


## 8. Optional spatial + UMAP visualization

When `RUN_SINGLE_BATCH_VIS` is True: one raw slice, `obs[COLOR_KEY]` as domain label, exports under `<output_path>/domain/`.


In [ ]:
if RUN_SINGLE_BATCH_VIS:
    # First raw h5ad in batch folder; domain-colored spatial + UMAP via MetricsComputer
    h5ads = sorted(raw_path.glob("*.h5ad"))
    if not h5ads:
        raise FileNotFoundError(f"No .h5ad files found under {raw_path}")

    adata = ad.read_h5ad(h5ads[0])
    if COLOR_KEY not in adata.obs.columns:
        raise ValueError(f"obs[{COLOR_KEY!r}] not found; cannot color region plots.")

    if getattr(adata, "isbacked", False):
        adata = adata.to_memory()
    else:
        adata = adata.copy()

    adata.obs["cluster"] = adata.obs[COLOR_KEY].astype(str)

    if "feat" in adata.obsm:
        feature_key = "feat"
    elif "X_pca" in adata.obsm:
        feature_key = "X_pca"
    else:
        raise ValueError(
            "Missing features for UMAP export. Expected adata.obsm['feat'] "
            "(preferred) or adata.obsm['X_pca']."
        )

    MetricsComputer.run_region_umap_metrics_export(
        adata,
        save_dir=output_path,
        feature_key=feature_key,
        cluster_key="cluster",
    )

    print("Visualization export finished.")
else:
    print("Skip optional spatial / UMAP export.")


## 9. Optional cross-stage time-series analysis

`TIME_MODE` selects development vs regeneration stage order; `WNT7B–FZD5` in `telencephalon` (see following cells).


In [ ]:
# Symlink per-stage CCI CSVs into <stage>/telencephalon/ and mirror filtered RNA paths for TimeSequenceAnalysis.


def ensure_time_sequence_cci_layout(cci_root: Path, stages, organ: str = "telencephalon"):
    for stage in stages:
        stage_dir = cci_root / stage
        if not stage_dir.is_dir():
            continue
        organ_dir = stage_dir / organ
        organ_dir.mkdir(parents=True, exist_ok=True)
        for f in stage_dir.glob("*.csv"):
            if not f.is_file():
                continue
            if f.name.upper() == "LR.CSV":
                continue
            link = organ_dir / f.name
            if link.exists() or link.is_symlink():
                continue
            try:
                link.symlink_to(f.resolve())
            except OSError:
                pass


def ensure_time_sequence_spatial_layout(spatial_root: Path, stages, organ: str = "telencephalon"):
    for stage in stages:
        src = spatial_root / stage / "preprocess" / "preprocessed_RNA.h5ad"
        if not src.is_file():
            continue
        target_dir = spatial_root / stage / "preprocess" / organ
        target_dir.mkdir(parents=True, exist_ok=True)
        dst = target_dir / "preprocessed_RNA_filtered.h5ad"
        if dst.exists() or dst.is_symlink():
            continue
        try:
            dst.symlink_to(src.resolve())
        except OSError:
            pass


# Ordered batch folders and CCI aggregate root for the selected program
if TIME_MODE == "development":
    batch_list = [
        "develop_44",
        "develop_54",
        "develop_57",
        "develop_Juvenile",
        "develop_Adult",
        "develop_Metamorphosed",
    ]
    time_work_dir = project_root / "data" / "axolotl_develop_cci"
elif TIME_MODE == "regeneration":
    batch_list = [
        "regeneration_2",
        "regeneration_5",
        "regeneration_10",
        "regeneration_15",
        "regeneration_20",
        "regeneration_30",
        "regeneration_60",
        "regeneration_control",
    ]
    time_work_dir = project_root / "data" / "axolotl_regeneration_cci"
else:
    raise ValueError("TIME_MODE must be 'development' or 'regeneration'")

time_output_path = time_work_dir / "output"
time_raw_path = time_work_dir / "raw"
time_output_path.mkdir(parents=True, exist_ok=True)

print("TIME_MODE =", TIME_MODE)
print("batch_list =", batch_list)
print("time_work_dir =", time_work_dir)


In [ ]:
if RUN_TIME_SERIES_ANALYSIS:
    # Prepare folder layout expected by TimeSequenceAnalysis (CCI + spatial preprocess)
    ensure_time_sequence_cci_layout(time_raw_path, batch_list, organ="telencephalon")
    ensure_time_sequence_spatial_layout(dataset_root, batch_list, organ="telencephalon")

    # Region and LR identifiers (hyphenated ligand-receptor names)
    lr_filter = ["WNT7B-FZD5"]
    annotation_filter = ["telencephalon"]
    region_lr_map = {"telencephalon": "WNT7B-FZD5"}

    # Multi-stage runner over batch_list
    time_sequence_analysis = TimeSequenceAnalysis(
        stages=batch_list,
        raw_path=time_raw_path,
        spatial_root=str(dataset_root),
        output_path=time_output_path,
        annotation_filter=annotation_filter,
        lr_filter=lr_filter,
        fig_format="svg",
        cell_type_key="Annotation",
    )

    # Cell counts per stage (bar / line export)
    time_sequence_analysis.count_cell_number(recompute=False, font_size=12)
    # CCC edge counts per stage
    time_sequence_analysis.count_edge_number(recompute=False, font_size=12)
    # Precision / recall style efficiency curves across stages
    time_sequence_analysis.plot_efficiency_metrics(
        recompute=False,
        region_lr_map=region_lr_map,
        font_size=8,
    )
    # Mean LR strength bars by cell type (per region key in region_lr_map)
    time_sequence_analysis.plot_celltype_strength_bars(
        recompute=False,
        region_lr_map=region_lr_map,
        font_size=8,
    )
    # Graph density (edges per cell) trajectories across stages
    time_sequence_analysis.plot_graph_density_over_stages(
        recompute=False,
        font_size=12,
    )
    # Strongest communicating nodes over stages
    time_sequence_analysis.plot_strong_nodes_over_stages(
        recompute=False,
        region_lr_map=region_lr_map,
        font_size=8,
    )
    # LR strength vs spatial distance across stages
    time_sequence_analysis.plot_strength_vs_distance_over_stages(
        recompute=False,
        region_lr_map=region_lr_map,
        font_size=12,
    )
    # CCDF of degree-weighted edge strength (per region LR)
    time_sequence_analysis.plot_ccdf_degree_strength(
        recompute=False,
        region_lr_map=region_lr_map,
        font_size=8,
    )

    print(f"Time-series analysis completed for TIME_MODE={TIME_MODE!r}")
    print(f"Time-series outputs saved under: {time_output_path}")
else:
    print("Skip time-series analysis.")
